# Differentiable Neural Computer

The DNC (Graves et al., 2016) extends the NTM with three key improvements:

1. **Usage-based allocation**: tracks which memory rows are free, allocates
   the least-used row for writes (no shift-based addressing needed)
2. **Temporal link matrix**: records the order in which memory was written,
   enabling forward/backward traversal of stored sequences
3. **Multi-head reads with mode mixture**: each read head blends content-based,
   forward-link, and backward-link addressing via learned mode weights

**Reference:** Graves et al., "Hybrid computing using a neural network
with dynamic external memory" (Nature, 2016)

**CLI equivalents:** `make example-dnc-copy` and `make example-dnc-recall`

## Architecture

The DNC has an additional type parameter `r` for the number of read heads.
Each read head can independently address memory using different modes.

In [ ]:
:t dncLayer

In [ ]:
:t DncState

## NTM vs DNC

| Feature | NTM | DNC |
|---------|-----|-----|
| Write addressing | Content + shift | Usage-based allocation |
| Read addressing | Content + shift | Content + temporal links |
| Memory management | None (overwrite) | Free gates + usage tracking |
| Read heads | 1 | R (type parameter) |
| Link matrix | None | O(N^2) temporal ordering |

The DNC is more powerful but slower per step (link matrix is O(N^2)).
Default N=32 (vs NTM's N=128) to keep the link matrix manageable.

## Training Demo

Same copy task as the NTM, but using the DNC architecture with
R=1 read head, N=10 memory rows, M=5 memory width.

This cell takes ~20-30 seconds due to the link matrix overhead.

In [ ]:
:exec do { srand 42;
  dnc <- dncLayer {ty = Variable CPU, r=1, inputSize=9, outputSize=8, n=10, m=5, h=20};
  model <- pure (autoName (OutputLayer dnc));
  putStrLn ("Model: " ++ show model);
  opt <- pure (nativeRmsprop 0.0001 0.95 1.0e-8 10.0 0.9);
  (trained, epochs, loss) <- runTraining
    (\m, d => epochTwoPhaseTensor opt d m)
    (copyTaskBinaryBatchVect {w=8} 1 1 5)
    (simpleConfig 500) model;
  putStrLn ("");
  putStrLn ("Trained " ++ show epochs ++ " epochs, final loss: " ++ show loss) }

## DNC Memory Mechanics

### Allocation
The DNC tracks *usage* of each memory row. When writing, it allocates
the least-used row via argsort + cumprod. Free gates allow the controller
to explicitly release memory rows after reading.

### Temporal Links
The link matrix `L[i,j]` records "row i was written immediately after row j".
This enables:
- **Forward read**: follow the write order (L * w_prev)
- **Backward read**: reverse the write order (L^T * w_prev)

### Read Modes
Each read head has 3 mode weights (softmax):
- Backward link weighting
- Content-based weighting
- Forward link weighting

The controller learns which mode to use for each head at each timestep.

## Scaling Up

For full convergence:
```bash
make example-dnc-copy --epochs 50000 --lr 0.0001
make example-dnc-recall --epochs 50000 --lr 0.0001
```

The DNC can also be configured with multiple read heads:
```bash
# R=4 matches the original paper but needs more training
```

R=1 exercises all DNC mechanisms. R=4 adds capacity but requires
more epochs to converge.

## PyTorch Comparison

The DNC is among the most complex architectures in the library.
The PyTorch implementation requires careful numerical clamping
at 6 points to prevent NaN during multi-timestep forward passes.
idris-ml handles this identically in the C backend.

See `pytorch/torch_ref/scripts/dnc_copy.py` and `dnc_recall.py`
for the full references.

Next: [CNN](cnn.ipynb) — convolutional networks for image classification.